# FASE 1. COMPRENSIÓN DE LOS DATOS - CALIDAD DEL AIRE (GVA)

## 1. Carga y observaciones iniciales

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TFM_Aire_GVA") \
    .getOrCreate()

# Ruta provisional de GVA
ruta_gva = "gs://tfm_unir1/aire/gva_quality/"

# Acceder a Google Cloud Storage
path = spark._jvm.org.apache.hadoop.fs.Path(ruta_gva)
fs = path.getFileSystem(spark._jsc.hadoopConfiguration())

# Listar el contenido
archivos = fs.listStatus(path)

print("Elementos encontrados:", len(archivos))

for archivo in archivos:
    print(archivo.getPath().getName())

Elementos encontrados: 6
gva_air_quality_2020.parquet
gva_air_quality_2021.parquet
gva_air_quality_2022.parquet
gva_air_quality_2023.parquet
gva_air_quality_2024.parquet
gva_air_quality_2025.parquet


In [3]:
# Leer conjuntamente los seis archivos
df_gva = spark.read.parquet(ruta_gva)

# Mostrar la estructura
df_gva.printSchema()

# Mostrar las primeras cinco observaciones
df_gva.show(5, truncate=False)

root
 |-- stream: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- station_code: string (nullable = true)
 |-- station_name: string (nullable = true)
 |-- date: string (nullable = true)
 |-- hour: long (nullable = true)
 |-- datetime: timestamp (nullable = true)
 |-- so2: long (nullable = true)
 |-- co: double (nullable = true)
 |-- no: long (nullable = true)
 |-- no2: long (nullable = true)
 |-- nox: long (nullable = true)
 |-- o3: long (nullable = true)
 |-- pm10: long (nullable = true)
 |-- pm2_5: long (nullable = true)
 |-- pm1: long (nullable = true)
 |-- nh3: long (nullable = true)
 |-- benzene_c6h6: double (nullable = true)
 |-- toluene_c7h8: double (nullable = true)
 |-- xylene_c8h10: double (nullable = true)
 |-- wind_direction_deg: long (nullable = true)
 |-- relative_humidity: long (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- pressure: long (nullable = true)
 |-- solar_radiation: long (nullable = true)
 |-- noise: long (nullable

26/10/21 12:18:41 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------------+-------------------------+------------+-------------------------+----------+----+-------------------+---+---+---+---+---+---+----+-----+----+----+------------+------------+------------+------------------+-----------------+-------------+--------+---------------+-----+-----------+----+----------+--------------+----+----+----+----+----+----+----+----+----+----+----+----+----+----+----+
|stream         |source_file              |station_code|station_name             |date      |hour|datetime           |so2|co |no |no2|nox|o3 |pm10|pm2_5|pm1 |nh3 |benzene_c6h6|toluene_c7h8|xylene_c8h10|wind_direction_deg|relative_humidity|precipitation|pressure|solar_radiation|noise|temperature|uv_b|wind_speed|wind_speed_max|as  |baa |bap |bbfa|bjfa|bkfa|cd  |daha|fa  |hmn |h2s |icdp|ni  |pb  |pst |
+---------------+-------------------------+------------+-------------------------+----------+----+-------------------+---+---+---+---+---+---+----+-----+----+----+------------+------------+-----

## 2. Caracterización general del conjunto GVA

### 2.1. Volumen de registros y cobertura temporal

In [4]:
from pyspark.sql import functions as F

df_gva.agg(
    F.count("*").alias("total_registros"),
    F.countDistinct("station_code").alias("total_estaciones"),
    F.min("datetime").alias("primera_medicion"),
    F.max("datetime").alias("ultima_medicion")
).show(truncate=False)

+---------------+----------------+-------------------+-------------------+
|total_registros|total_estaciones|primera_medicion   |ultima_medicion    |
+---------------+----------------+-------------------+-------------------+
|464088         |9               |2020-01-01 00:00:00|2025-12-31 23:00:00|
+---------------+----------------+-------------------+-------------------+



### 2.2. Identificación de estaciones de medición

In [5]:
from pyspark.sql import functions as F

df_gva.groupBy(
    "station_code",
    "station_name"
).agg(
    F.count("*").alias("registros"),
    F.min("datetime").alias("primera_medicion"),
    F.max("datetime").alias("ultima_medicion")
).orderBy("station_code").show(20, truncate=False)

+------------+---------------------------+---------+-------------------+-------------------+
|station_code|station_name               |registros|primera_medicion   |ultima_medicion    |
+------------+---------------------------+---------+-------------------+-------------------+
|46250030    |VALÈNCIA - PISTA DE SILLA  |51888    |2020-01-01 00:00:00|2025-12-31 23:00:00|
|46250043    |VALÈNCIA - VIVERS          |51888    |2020-01-01 00:00:00|2025-12-31 23:00:00|
|46250046    |VALÈNCIA - POLITÈCNIC      |51888    |2020-01-01 00:00:00|2025-12-31 23:00:00|
|46250047    |VALÈNCIA - AVD. FRANCIA    |51888    |2020-01-01 00:00:00|2025-12-31 23:00:00|
|46250048    |VALÈNCIA - MOLÍ DEL SOL    |51888    |2020-01-01 00:00:00|2025-12-31 23:00:00|
|46250049    |VALÈNCIA-CONSELLERIA METEO.|51888    |2020-01-01 00:00:00|2025-12-31 23:00:00|
|46250050    |VALÈNCIA - BULEVARD SUD    |51888    |2020-01-01 00:00:00|2025-12-31 23:00:00|
|46250054    |VALÈNCIA CENTRE            |51888    |2020-01-01 00:00:0

### 2.3. Disponibilidad de contaminantes por estación

In [6]:
from pyspark.sql import functions as F

df_gva.groupBy(
    "station_code",
    "station_name"
).agg(
    F.count("no2").alias("registros_NO2"),
    F.count("o3").alias("registros_O3"),
    F.count("pm10").alias("registros_PM10"),
    F.count("pm2_5").alias("registros_PM25")
).orderBy("station_code").show(20, truncate=False)

+------------+---------------------------+-------------+------------+--------------+--------------+
|station_code|station_name               |registros_NO2|registros_O3|registros_PM10|registros_PM25|
+------------+---------------------------+-------------+------------+--------------+--------------+
|46250030    |VALÈNCIA - PISTA DE SILLA  |51079        |48021       |50812         |50813         |
|46250043    |VALÈNCIA - VIVERS          |50856        |50523       |0             |0             |
|46250046    |VALÈNCIA - POLITÈCNIC      |49059        |49032       |49076         |49076         |
|46250047    |VALÈNCIA - AVD. FRANCIA    |51114        |50291       |49103         |48935         |
|46250048    |VALÈNCIA - MOLÍ DEL SOL    |48953        |50273       |49908         |49908         |
|46250049    |VALÈNCIA-CONSELLERIA METEO.|0            |0           |0             |0             |
|46250050    |VALÈNCIA - BULEVARD SUD    |50295        |48987       |0             |0             |


1. Cuatro estaciones contienen los cuatro contaminantes: Pista de Silla, Politècnic, Avd. França y Molí del Sol. Son candidatas interesantes para estudiar modelos multivariables.
2. Dos estaciones no presentan mediciones de los cuatro contaminantes seleccionados: Conselleria Meteo. y Nazaret MET-2. No debemos descartarlas, porque podrían aportar variables meteorológicas útiles para la predicción.
3. GVA podría complementar EEA, especialmente porque incorpora variables meteorológicas junto a las concentraciones. No obstante, debemos comprobar si las mediciones de contaminación de ambas fuentes coinciden.

## 3. Evaluación de la calidad de los datos GVA

### 3.1. Identificación de valores negativos y especiales

In [8]:
from pyspark.sql import functions as F

contaminantes_gva = ["no2", "o3", "pm10", "pm2_5"]

df_gva.agg(
    *[F.sum(
            F.when(F.col(c) < 0, 1).otherwise(0)
        ).alias(f"{c}_negativos")
        for c in contaminantes_gva],
    *[F.min(c).alias(f"{c}_minimo")
        for c in contaminantes_gva],
    *[F.max(c).alias(f"{c}_maximo")
        for c in contaminantes_gva]
).show(truncate=False)

+-------------+------------+--------------+---------------+----------+---------+-----------+------------+----------+---------+-----------+------------+
|no2_negativos|o3_negativos|pm10_negativos|pm2_5_negativos|no2_minimo|o3_minimo|pm10_minimo|pm2_5_minimo|no2_maximo|o3_maximo|pm10_maximo|pm2_5_maximo|
+-------------+------------+--------------+---------------+----------+---------+-----------+------------+----------+---------+-----------+------------+
|0            |0           |0             |0              |1         |1        |1          |1           |169       |169      |831        |652         |
+-------------+------------+--------------+---------------+----------+---------+-----------+------------+----------+---------+-----------+------------+



- No aparecen valores negativos, por lo que no hemos encontrado códigos como -999 en las concentraciones no nulas.
- El mínimo es 1 para los cuatro contaminantes

### 3.2. Comprobación de registros duplicados

In [10]:
from pyspark.sql import functions as F

duplicados_gva = (
    df_gva
    .groupBy("station_code", "datetime")
    .agg(F.count("*").alias("repeticiones"))
    .filter(F.col("repeticiones") > 1))

duplicados_gva.agg(
    F.count("*").alias("combinaciones_duplicadas"),
    F.coalesce(
        F.sum(F.col("repeticiones") - 1),
        F.lit(0)
    ).alias("registros_excedentes")).show()

+------------------------+--------------------+
|combinaciones_duplicadas|registros_excedentes|
+------------------------+--------------------+
|                       0|                   0|
+------------------------+--------------------+



## 4. Análisis de cobertura temporal

### 4.1. Disponibilidad de registros por estación y año

In [12]:
from pyspark.sql import functions as F

df_gva.groupBy(
    "station_code",F.year("datetime").alias("anio")
).agg(F.count("*").alias("horas_registradas")
).orderBy("station_code", "anio"
).show(60, truncate=False)

+------------+----+-----------------+
|station_code|anio|horas_registradas|
+------------+----+-----------------+
|46250030    |2020|8784             |
|46250030    |2021|8760             |
|46250030    |2022|8760             |
|46250030    |2023|8040             |
|46250030    |2024|8784             |
|46250030    |2025|8760             |
|46250043    |2020|8784             |
|46250043    |2021|8760             |
|46250043    |2022|8760             |
|46250043    |2023|8040             |
|46250043    |2024|8784             |
|46250043    |2025|8760             |
|46250046    |2020|8784             |
|46250046    |2021|8760             |
|46250046    |2022|8760             |
|46250046    |2023|8040             |
|46250046    |2024|8784             |
|46250046    |2025|8760             |
|46250047    |2020|8784             |
|46250047    |2021|8760             |
|46250047    |2022|8760             |
|46250047    |2023|8040             |
|46250047    |2024|8784             |
|46250047   

En 2023 faltan exactamente 720 horas en todas las estaciones. Esto sugiere que podría faltar un periodo completo de 30 días en el archivo de 2023, ya que 720 horas equivalen a 30 días. Todavía debemos comprobarlo.

La estación 46250900 presenta una segunda particularidad:
- En 2020 tiene 5.880 registros, frente a las 8.784 horas esperadas.
- En 2023 también presenta 8.040 registros.
- En los demás años tiene el número esperado de filas.

### 4.2. Identificación de periodos sin registros

In [14]:
from pyspark.sql import functions as F

# Generar todas las horas esperadas de 2023
horas_esperadas = spark.sql("""
    SELECT explode(
        sequence(
            timestamp('2023-01-01 00:00:00'),
            timestamp('2023-12-31 23:00:00'),
            interval 1 hour)) AS datetime""")

# Horas disponibles en Pista de Silla
horas_reales = (
    df_gva.filter(
        (F.col("station_code") == "46250030") &
        (F.year("datetime") == 2023))
    .select("datetime")
    .distinct())

# Identificar las horas que faltan
horas_ausentes = horas_esperadas.join(
    horas_reales,
    on="datetime",
    how="left_anti")

# Agrupar las ausencias por día
horas_ausentes.groupBy(
    F.to_date("datetime").alias("fecha")
).agg(F.count("*").alias("horas_ausentes")
).orderBy("fecha").show(40, truncate=False)

+----------+--------------+
|fecha     |horas_ausentes|
+----------+--------------+
|2023-09-01|24            |
|2023-09-02|24            |
|2023-09-03|24            |
|2023-09-04|24            |
|2023-09-05|24            |
|2023-09-06|24            |
|2023-09-07|24            |
|2023-09-08|24            |
|2023-09-09|24            |
|2023-09-10|24            |
|2023-09-11|24            |
|2023-09-12|24            |
|2023-09-13|24            |
|2023-09-14|24            |
|2023-09-15|24            |
|2023-09-16|24            |
|2023-09-17|24            |
|2023-09-18|24            |
|2023-09-19|24            |
|2023-09-20|24            |
|2023-09-21|24            |
|2023-09-22|24            |
|2023-09-23|24            |
|2023-09-24|24            |
|2023-09-25|24            |
|2023-09-26|24            |
|2023-09-27|24            |
|2023-09-28|24            |
|2023-09-29|24            |
|2023-09-30|24            |
+----------+--------------+



Faltan los 30 días completos de septiembre de 2023 en los datos de GVA de Pista de Silla
Si entrenamos un modelo horario sin considerar este hueco, podríamos construir incorrectamente variables retardadas.
Por ejemplo, la última observación de agosto podría quedar situada inmediatamente antes de la primera de octubre al ordenar las filas, aunque entre ambas haya transcurrido un mes.
Por ello, tendremos que conservar la continuidad del calendario y representar explícitamente los periodos sin información, sin inventar concentraciones.

### 4.3. Verificación de ausencias temporales comunes entre estaciones

In [16]:
from pyspark.sql import functions as F

df_gva.groupBy(
    "station_code",
    "station_name"
).agg(F.sum(
        F.when(
            (F.col("datetime") >= "2023-09-01 00:00:00") &
            (F.col("datetime") < "2023-10-01 00:00:00"),1).otherwise(0)
    ).alias("registros_septiembre_2023")
).orderBy("station_code").show(truncate=False)

+------------+---------------------------+-------------------------+
|station_code|station_name               |registros_septiembre_2023|
+------------+---------------------------+-------------------------+
|46250030    |VALÈNCIA - PISTA DE SILLA  |0                        |
|46250043    |VALÈNCIA - VIVERS          |0                        |
|46250046    |VALÈNCIA - POLITÈCNIC      |0                        |
|46250047    |VALÈNCIA - AVD. FRANCIA    |0                        |
|46250048    |VALÈNCIA - MOLÍ DEL SOL    |0                        |
|46250049    |VALÈNCIA-CONSELLERIA METEO.|0                        |
|46250050    |VALÈNCIA - BULEVARD SUD    |0                        |
|46250054    |VALÈNCIA CENTRE            |0                        |
|46250900    |VALÈNCIA - NAZARET MET-2   |0                        |
+------------+---------------------------+-------------------------+



Las nueve estaciones GVA carecen de registros durante todo septiembre de 2023.

## 5. Disponibilidad de variables meteorológicas

### 5.1. Cobertura de las principales variables explicativas

In [18]:
from pyspark.sql import functions as F

variables_meteo = [
    "temperature",
    "relative_humidity",
    "wind_speed",
    "wind_direction_deg",
    "precipitation",
    "pressure",
    "solar_radiation"]

df_gva.groupBy("station_code"
).agg(F.count("*").alias("total_registros"),
    *[F.round(F.count(c) / F.count("*") * 100, 2
        ).alias(f"{c}_pct")
        for c in variables_meteo]
).orderBy("station_code").show(20, truncate=False)

+------------+---------------+---------------+---------------------+--------------+----------------------+-----------------+------------+-------------------+
|station_code|total_registros|temperature_pct|relative_humidity_pct|wind_speed_pct|wind_direction_deg_pct|precipitation_pct|pressure_pct|solar_radiation_pct|
+------------+---------------+---------------+---------------------+--------------+----------------------+-----------------+------------+-------------------+
|46250030    |51888          |0.0            |0.0                  |0.0           |0.0                   |0.0              |0.0         |0.0                |
|46250043    |51888          |0.0            |0.0                  |0.0           |0.0                   |0.0              |0.0         |0.0                |
|46250046    |51888          |0.0            |0.0                  |0.0           |0.0                   |0.0              |0.0         |0.0                |
|46250047    |51888          |0.0            |0.0   

1. Tenemos dos estaciones principalmente meteorológicas
- Conselleria Meteo. (46250049)
- Nazaret MET-2 (46250900)

Ambas presentan una cobertura elevada de temperatura, humedad, velocidad del viento, precipitación, presión y radiación solar, considerando los registros existentes.

2. Avd. França dispone de información de viento: Aunque no tiene temperatura ni humedad, presenta un 97,50 % de disponibilidad de velocidad del viento y un 98,51 % de dirección.
3. Las demás estaciones no tienen estas variables meteorológicas informadas.

## 6. Disponibilidad de datos de ruido

### 6.1. Comprobar los tipos de datos de noise

In [21]:
# Comprobar el tipo de dato de noise en cada año

for anio in range(2020, 2026):
    ruta = f"gs://tfm_unir1/aire/gva_quality/gva_air_quality_{anio}.parquet"
    df_anio = spark.read.parquet(ruta)
    print(f"Año {anio}: {df_anio.schema['noise'].dataType}")

Año 2020: LongType()
Año 2021: LongType()
Año 2022: LongType()
Año 2023: DoubleType()
Año 2024: DoubleType()
Año 2025: DoubleType()


### 6.2. Armonización del tipo de dato de ruido para su análisis

In [22]:
from pyspark.sql import functions as F
from functools import reduce

# Leer cada año y unificar el tipo de noise
dataframes = []

for anio in range(2020, 2026):
    ruta = f"gs://tfm_unir1/aire/gva_quality/gva_air_quality_{anio}.parquet"

    df_anio = (spark.read.parquet(ruta)
        .withColumn("noise", F.col("noise").cast("double")))

    dataframes.append(df_anio)

# Unir los seis años
df_gva_unificado = reduce(
    lambda a, b: a.unionByName(b),
    dataframes)

# Comprobar el resultado
print("Tipo de noise:", df_gva_unificado.schema["noise"].dataType)

print("Registros:", df_gva_unificado.count())

Tipo de noise: DoubleType()


Registros: 464088


### 6.3. Disponibilidad de mediciones de ruido por estación

In [23]:
from pyspark.sql import functions as F

df_gva_unificado.groupBy(
    "station_code",
    "station_name"
).agg(
    F.count("*").alias("total_registros"),
    F.count("noise").alias("registros_ruido"),
    F.round(
        F.count("noise") / F.count("*") * 100, 2
    ).alias("cobertura_ruido_pct")
).orderBy("station_code").show(20, truncate=False)

+------------+---------------------------+---------------+---------------+-------------------+
|station_code|station_name               |total_registros|registros_ruido|cobertura_ruido_pct|
+------------+---------------------------+---------------+---------------+-------------------+
|46250030    |VALÈNCIA - PISTA DE SILLA  |51888          |23648          |45.58              |
|46250043    |VALÈNCIA - VIVERS          |51888          |0              |0.0                |
|46250046    |VALÈNCIA - POLITÈCNIC      |51888          |0              |0.0                |
|46250047    |VALÈNCIA - AVD. FRANCIA    |51888          |0              |0.0                |
|46250048    |VALÈNCIA - MOLÍ DEL SOL    |51888          |0              |0.0                |
|46250049    |VALÈNCIA-CONSELLERIA METEO.|51888          |0              |0.0                |
|46250050    |VALÈNCIA - BULEVARD SUD    |51888          |0              |0.0                |
|46250054    |VALÈNCIA CENTRE            |51888   

Solo Pista de Silla dispone de mediciones de ruido.

### 6.4. Distribución anual de las mediciones de ruido

In [24]:
from pyspark.sql import functions as F

df_gva_unificado.filter(
    F.col("station_code") == "46250030"
).groupBy(
    F.year("datetime").alias("anio")
).agg(
    F.count("*").alias("total_registros"),
    F.count("noise").alias("registros_ruido"),
    F.round(
        F.count("noise") / F.count("*") * 100, 2
    ).alias("cobertura_ruido_pct")
).orderBy("anio").show()

+----+---------------+---------------+-------------------+
|anio|total_registros|registros_ruido|cobertura_ruido_pct|
+----+---------------+---------------+-------------------+
|2020|           8784|           8561|              97.46|
|2021|           8760|           8544|              97.53|
|2022|           8760|           6543|              74.69|
|2023|           8040|              0|                0.0|
|2024|           8784|              0|                0.0|
|2025|           8760|              0|                0.0|
+----+---------------+---------------+-------------------+



Los años 2020 y 2021 presentan una disponibilidad muy elevada de ruido en Pista de Silla. En 2022 disminuye, y desde 2023 no tenemos mediciones disponibles en esta columna.

Por tanto, no utilizaría noise de GVA como variable predictora general para todas las estaciones y todo el periodo 2020–2025.

## 7. Cobertura temporal de los contaminantes

### 7.1. Disponibilidad anual de las mediciones

In [25]:
from pyspark.sql import functions as F

df_gva_unificado.groupBy(
    "station_code",
    F.year("datetime").alias("anio")
).agg(
    F.round(F.count("no2") / F.count("*") * 100, 2)
     .alias("NO2_pct"),
    F.round(F.count("o3") / F.count("*") * 100, 2)
     .alias("O3_pct"),
    F.round(F.count("pm10") / F.count("*") * 100, 2)
     .alias("PM10_pct"),
    F.round(F.count("pm2_5") / F.count("*") * 100, 2)
     .alias("PM25_pct")
).orderBy("station_code", "anio").show(60)

+------------+----+-------+------+--------+--------+
|station_code|anio|NO2_pct|O3_pct|PM10_pct|PM25_pct|
+------------+----+-------+------+--------+--------+
|    46250030|2020|  97.12| 88.55|   97.55|   97.55|
|    46250030|2021|   96.2| 87.72|   97.24|   97.24|
|    46250030|2022|  99.44| 91.44|   97.91|   97.92|
|    46250030|2023|  99.42| 98.98|   99.93|   99.93|
|    46250030|2024|   99.4| 99.17|   95.38|   95.38|
|    46250030|2025|  99.16| 89.95|   99.73|   99.73|
|    46250043|2020|  98.42| 97.46|     0.0|     0.0|
|    46250043|2021|  97.26| 94.53|     0.0|     0.0|
|    46250043|2022|  99.14| 98.82|     0.0|     0.0|
|    46250043|2023|  99.39| 98.51|     0.0|     0.0|
|    46250043|2024|  97.27| 99.43|     0.0|     0.0|
|    46250043|2025|   96.7| 95.55|     0.0|     0.0|
|    46250046|2020|  98.86| 95.07|   95.64|   95.64|
|    46250046|2021|  94.26| 93.49|   90.64|   90.64|
|    46250046|2022|  87.24| 93.32|   92.55|   92.55|
|    46250046|2023|   95.5|  96.8|   95.04|   

1. Las estaciones con los cuatro contaminantes presentan una cobertura elevada.
2. El NO₂ presenta una buena disponibilidad.
3. La disponibilidad del ozono también es elevada, pero varía por estación.
4. PM10 y PM2.5 tienen coberturas prácticamente idénticas.
5. Las estaciones meteorológicas no aportan directamente los cuatro contaminantes seleccionados.

## 8. Comparación preliminar entre las fuentes GVA y EEA

### 8.1. Identificación de correspondencias entre estaciones

In [27]:
# Cargar los datos horarios preparados de EEA
ruta_eea_limpio = "gs://tfm_unir1/processed/aire/eea/horario/"

df_eea_limpio = spark.read.parquet(ruta_eea_limpio)

# Examinar los identificadores de los puntos de medición
df_eea_limpio.select(
    "id_punto_medicion",
    "codigo_contaminante",
    "contaminante"
).distinct().orderBy(
    "id_punto_medicion"
).show(50, truncate=False)

+--------------------+-------------------+------------+
|id_punto_medicion   |codigo_contaminante|contaminante|
+--------------------+-------------------+------------+
|ES/SP_46078004_14_6 |7                  |O3          |
|ES/SP_46078004_8_8  |8                  |NO2         |
|ES/SP_46102002_10_46|5                  |PM10        |
|ES/SP_46102002_14_6 |7                  |O3          |
|ES/SP_46102002_8_8  |8                  |NO2         |
|ES/SP_46102002_9_46 |6001               |PM2.5       |
|ES/SP_46190005_14_6 |7                  |O3          |
|ES/SP_46190005_8_8  |8                  |NO2         |
|ES/SP_46244003_10_46|5                  |PM10        |
|ES/SP_46244003_14_6 |7                  |O3          |
|ES/SP_46244003_8_8  |8                  |NO2         |
|ES/SP_46244003_9_46 |6001               |PM2.5       |
|ES/SP_46250030_10_46|5                  |PM10        |
|ES/SP_46250030_14_6 |7                  |O3          |
|ES/SP_46250030_8_8  |8                  |NO2   

Estos identificadores indican que las dos fuentes hacen referencia a los mismos códigos de estación.

Sin embargo, tener el mismo código no garantiza que las concentraciones coincidan exactamente. Para comprobarlo necesitamos comparar las observaciones.

### 8.2. Comparación global de mediciones entre GVA y EEA

In [29]:
from pyspark.sql import functions as F

# 1. Preparar EEA
# Extraer el código de estación del identificador EEA

eea_comp = (
    df_eea_limpio
    .withColumn(
        "station_code",
        F.regexp_extract(
            F.col("id_punto_medicion"),r"SP_(\d{8})_",1))
    .select("station_code",
        F.col("fecha_inicio").alias("fecha"),"contaminante",
        F.col("concentracion").cast("double").alias("valor_eea")))

# 2. Preparar GVA
# Convertir los cuatro contaminantes a formato largo

gva_comp = (
    df_gva_unificado
    .select("station_code",
        F.col("datetime").alias("fecha"),
        F.expr("""stack(
                4,
                'NO2', cast(no2 as double),
                'O3', cast(o3 as double),
                'PM10', cast(pm10 as double),
                'PM2.5', cast(pm2_5 as double)
            ) as (contaminante, valor_gva)""")))

# 3. Comparar ambas fuentes
# Unir por estación, fecha y contaminante

comparacion_global = (
    gva_comp
    .join(eea_comp,["station_code", "fecha", "contaminante"],"inner")
    .filter(
        F.col("valor_gva").isNotNull() &
        F.col("valor_eea").isNotNull())
    .withColumn("diferencia",
        F.abs(F.col("valor_gva") - F.col("valor_eea"))))

# 4. Obtener estadísticas de coincidencia
resumen_comparacion = (
    comparacion_global
    .groupBy("station_code", "contaminante")
    .agg(F.count("*").alias("observaciones_comparadas"),
        F.sum(
            F.when(F.col("diferencia") == 0, 1).otherwise(0)
        ).alias("coincidencias_exactas"),
        F.round(
            F.avg(
                F.when(F.col("diferencia") == 0, 1).otherwise(0)
            ) * 100, 2).alias("coincidencia_pct"),
        F.round(
            F.avg("diferencia"), 3).alias("diferencia_media"),
        F.round(
            F.max("diferencia"), 3).alias("diferencia_maxima"))
    .orderBy("contaminante", "station_code"))

resumen_comparacion.show(50, truncate=False)

+------------+------------+------------------------+---------------------+----------------+----------------+-----------------+
|station_code|contaminante|observaciones_comparadas|coincidencias_exactas|coincidencia_pct|diferencia_media|diferencia_maxima|
+------------+------------+------------------------+---------------------+----------------+----------------+-----------------+
|46250030    |NO2         |50731                   |50393                |99.33           |0.046           |45.0             |
|46250043    |NO2         |50458                   |49932                |98.96           |0.062           |45.0             |
|46250046    |NO2         |48650                   |48331                |99.34           |0.055           |59.0             |
|46250047    |NO2         |50717                   |49340                |97.28           |0.081           |54.0             |
|46250048    |NO2         |48619                   |48205                |99.15           |0.057           |55.

1. NO₂ presenta una coincidencia especialmente elevada.
Todas las estaciones superan el 97 % de coincidencias exactas, llegando al 99,38 % en Bulevard Sud.
2. El ozono también muestra una elevada equivalencia.
Las coincidencias se sitúan aproximadamente entre el 97,68 % y el 98,61 %.
3. PM10 presenta mayores discrepancias.
4. PM2.5 presenta una situación similar.

Los resultados sugieren que GVA y EEA contienen, en gran medida, las mismas observaciones de contaminación.
Por tanto, no conviene concatenar ambas fuentes como si fueran mediciones independientes, porque estaríamos duplicando información.

Propuesta provisional: 
- EEA: como fuente principal de concentraciones con indicadores de validez y verificación.
- GVA: como fuente complementaria, especialmente para incorporar información meteorológica y estudiar la posibilidad de completar periodos sin mediciones en EEA.

Primero hay que comprobar que las diferencias no se deben a distintas referencias horarias, procesos de validación o revisiones de las mediciones.

### 8.3. Evolución anual de las diferencias entre GVA y EEA

In [30]:
from pyspark.sql import functions as F

comparacion_global.groupBy(
    F.year("fecha").alias("anio"),
    "contaminante"
).agg(
    F.count("*").alias("comparadas"),
    F.round(F.avg(
            F.when(F.col("diferencia") == 0, 1).otherwise(0)) * 100, 2
    ).alias("coincidencia_pct"),
    F.round(F.avg("diferencia"), 3
    ).alias("diferencia_media"),
    F.round(F.max("diferencia"), 2
    ).alias("diferencia_maxima")
).orderBy("contaminante", "anio").show(30, truncate=False)

+----+------------+----------+----------------+----------------+-----------------+
|anio|contaminante|comparadas|coincidencia_pct|diferencia_media|diferencia_maxima|
+----+------------+----------+----------------+----------------+-----------------+
|2020|NO2         |60173     |100.0           |0.0             |0.0              |
|2021|NO2         |59258     |99.99           |0.0             |2.0              |
|2022|NO2         |59379     |99.99           |0.0             |8.0              |
|2023|NO2         |54900     |100.0           |0.0             |0.0              |
|2024|NO2         |60223     |100.0           |0.0             |0.0              |
|2025|NO2         |55869     |93.44           |0.387           |80.0             |
|2020|O3          |50058     |100.0           |0.0             |7.0              |
|2021|O3          |48912     |99.98           |0.0             |2.0              |
|2022|O3          |49822     |99.99           |0.001           |24.0             |
|202

1. Entre 2020 y 2024 las mediciones coinciden casi completamente.
2. En 2025 las diferencias aumentan considerablemente.

### 8.4. Análisis mensual de las discrepancias de 2025

In [31]:
from pyspark.sql import functions as F

comparacion_global.filter(
    F.year("fecha") == 2025
).groupBy(
    F.month("fecha").alias("mes"),"contaminante"
).agg(
    F.count("*").alias("comparadas"),
    F.round(F.avg(
            F.when(F.col("diferencia") == 0, 1).otherwise(0)) * 100, 2
    ).alias("coincidencia_pct"),
    F.round(F.avg("diferencia"), 3).alias("diferencia_media")
).orderBy("contaminante", "mes").show(50, truncate=False)

+---+------------+----------+----------------+----------------+
|mes|contaminante|comparadas|coincidencia_pct|diferencia_media|
+---+------------+----------+----------------+----------------+
|1  |NO2         |4733      |94.74           |0.435           |
|2  |NO2         |4396      |94.38           |0.64            |
|3  |NO2         |4539      |91.5            |0.811           |
|4  |NO2         |4426      |93.31           |0.315           |
|5  |NO2         |4726      |93.91           |0.297           |
|6  |NO2         |4433      |93.82           |0.25            |
|7  |NO2         |4712      |93.72           |0.3             |
|8  |NO2         |4919      |93.13           |0.292           |
|9  |NO2         |4766      |92.87           |0.245           |
|10 |NO2         |4704      |93.54           |0.374           |
|11 |NO2         |4446      |93.36           |0.304           |
|12 |NO2         |5069      |93.02           |0.403           |
|1  |O3          |4314      |89.36      

NO₂: mantiene una coincidencia relativamente estable, entre el 91,50 % y el 94,74 %. Esto indica que las diferencias están distribuidas durante todo 2025.

O₃: presenta una anomalía especialmente llamativa en marzo, con solo un 41,82 % de coincidencias y una diferencia media de 2,635 µg/m³. En abril vuelve a superar el 95 %.

PM10: muestra una disminución progresiva de las coincidencias. Pasa del 81,12 % en enero al entorno del 51–53 % durante buena parte del segundo semestre.

PM2.5: presenta una coincidencia especialmente baja en febrero (35,37 %), mejora notablemente en abril (88,50 %) y posteriormente se estabiliza alrededor del 70 %.

### 8.5. Disponibilidad complementaria de mediciones GVA respecto a EEA

In [32]:
from pyspark.sql import functions as F

# Comparar la disponibilidad de ambas fuentes
# Conservamos todas las observaciones no nulas de GVA

disponibilidad = (
    gva_comp
    .filter(F.col("valor_gva").isNotNull())
    .join(eea_comp,["station_code", "fecha", "contaminante"],"left"))

# Identificar mediciones disponibles en GVA
# pero ausentes o inválidas en EEA

disponibilidad.groupBy("contaminante").agg(
    F.count("*").alias("registros_gva"),
    F.sum(F.when(F.col("valor_eea").isNull(), 1).otherwise(0)
    ).alias("sin_concentracion_valida_eea")
).orderBy("contaminante").show()

+------------+-------------+----------------------------+
|contaminante|registros_gva|sin_concentracion_valida_eea|
+------------+-------------+----------------------------+
|         NO2|       352382|                        2580|
|          O3|       297127|                        2130|
|        PM10|       248436|                       10496|
|       PM2.5|       248263|                       10687|
+------------+-------------+----------------------------+



GVA contiene mediciones de contaminación que no tienen una concentración válida correspondiente en EEA, al menos según las fechas y estaciones que hemos comparado.

1. GVA aporta potencialmente información complementaria.
2. Las partículas presentan una mayor complementariedad potencial.

### 8.6. Clasificación de las mediciones complementarias de GVA

In [33]:
from pyspark.sql import functions as F

# Partimos del DataFrame disponibilidad,
# creado en el paso anterior.

disponibilidad.groupBy("contaminante").agg(
    F.sum(F.when(F.col("valor_eea").isNull(), 1
        ).otherwise(0)).alias("sin_valor_valido_eea")
).orderBy("contaminante").show()

+------------+--------------------+
|contaminante|sin_valor_valido_eea|
+------------+--------------------+
|         NO2|                2580|
|          O3|                2130|
|        PM10|               10496|
|       PM2.5|               10687|
+------------+--------------------+



In [34]:
from pyspark.sql import functions as F

# Identificar si existe un registro en EEA
eea_disponibilidad = eea_comp.withColumn(
    "existe_eea", F.lit(1))

# Comparar los registros de GVA con EEA
comparacion_disponibilidad = (gva_comp
    .filter(F.col("valor_gva").isNotNull())
    .join(eea_disponibilidad,
        ["station_code", "fecha", "contaminante"],"left"))

# Clasificar las ausencias
comparacion_disponibilidad.groupBy("contaminante").agg(
    F.sum(
        F.when(
            F.col("existe_eea").isNull(), 1).otherwise(0)
    ).alias("sin_registro_eea"),
    F.sum(F.when(
            (F.col("existe_eea") == 1) &
            F.col("valor_eea").isNull(),1).otherwise(0)
    ).alias("registro_eea_invalido")
).orderBy("contaminante").show()

+------------+----------------+---------------------+
|contaminante|sin_registro_eea|registro_eea_invalido|
+------------+----------------+---------------------+
|         NO2|               0|                 2580|
|          O3|               0|                 2130|
|        PM10|            8631|                 1865|
|       PM2.5|            8625|                 2062|
+------------+----------------+---------------------+



1. NO₂ y O₃

Todas las observaciones GVA analizadas encuentran un registro correspondiente en EEA. Sin embargo, existen 2.580 registros de NO₂ y 2.130 de O₃ cuya concentración EEA no es utilizable.

2. PM10 y PM2.5

Aquí encontramos una diferencia importante: GVA dispone de 8.631 observaciones de PM10 y 8.625 de PM2.5 sin una fila EEA correspondiente.
Esto sugiere que GVA podría aportar cobertura adicional para las partículas.

3. Mediciones inválidas en EEA

Hay 8.637 observaciones GVA que coinciden con filas EEA cuya concentración limpia es NULL.
No debemos utilizarlas automáticamente para sustituir los valores inválidos, porque EEA podría haber descartado esas mediciones por motivos de calidad.

Con los análisis realizados, propongo mantener estas decisiones:
- EEA será nuestra fuente principal de concentraciones, gracias a sus indicadores de validez.
- GVA será una fuente complementaria de meteorología y posibles mediciones adicionales.
- No mezclaremos automáticamente concentraciones de ambas fuentes.
- Conservaremos la información de origen para mantener la trazabilidad de los datos.